# Mamba3学习笔记

## 导入包 pip install torch einops

In [1]:
import json
import math
from dataclasses import dataclass
from typing import Iterable, NamedTuple, TypeAlias, cast

import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange, repeat
from torch import LongTensor, Tensor, nn

## 使用说明

### 在ipynb里类与方法的绑定

In [2]:
class A:
    pass

In [3]:
def C(self, b=1):
    print(b)
A.C = C

In [4]:
# 创建实例
a = A()
# 通过实例调用，self 自动绑定为 a
a.C(5)

5


### 矩阵相乘

元素乘

In [5]:
a = torch.randn((1,2,3))
b = torch.randn((1,2,4))
a.shape, b.shape 

(torch.Size([1, 2, 3]), torch.Size([1, 2, 4]))

In [6]:
a.unsqueeze(-1).shape

torch.Size([1, 2, 3, 1])

In [7]:
rearrange(b, "b l n -> b l 1 n").shape

torch.Size([1, 2, 1, 4])

In [8]:
c = a.unsqueeze(-1) * rearrange(b, "b l n -> b l 1 n")
c.shape

torch.Size([1, 2, 3, 4])

矩阵乘

In [9]:
a = torch.randn((1,2,3))
b = torch.randn((1,4,3))
a.shape, b.shape

(torch.Size([1, 2, 3]), torch.Size([1, 4, 3]))

In [10]:
torch.einsum("bcd, bed -> bce", a, b).shape

torch.Size([1, 2, 4])

In [11]:
torch.einsum("bcd, bed -> bec", a, b).shape

torch.Size([1, 4, 2])

In [12]:
torch.einsum("bcd, bed -> dec", a, b).shape

torch.Size([3, 4, 2])

In [13]:
A = torch.tensor([[1.0],  # 形状: (a=2, b=1)
                  [4.0]])

B = torch.tensor([[0.1, 0.2, 0.3],  # 形状: (a=2, c=3)
                  [0.5, 0.6, 0.7]])
print("输入 A (形状: {}):".format(A.shape))
print("\n输入 B (形状: {}):".format(B.shape))
# 使用 einsum 计算
result = torch.einsum("ab, ac -> abc", A, B)
result

输入 A (形状: torch.Size([2, 1])):

输入 B (形状: torch.Size([2, 3])):


tensor([[[0.1000, 0.2000, 0.3000]],

        [[2.0000, 2.4000, 2.8000]]])

In [14]:
a = torch.randn((1,2,3))
b = torch.randn((4,3))
a,b

(tensor([[[ 0.7340, -0.2087,  0.5536],
          [ 0.4297,  0.0763, -0.2772]]]),
 tensor([[ 0.7491,  1.5529, -0.7985],
         [ 0.9210, -0.3693, -0.4365],
         [ 0.3822,  0.0564,  0.2036],
         [-0.2199, -0.6416, -0.1310]]))

In [15]:
rearrange(a, "b l n -> b l 1 n")

tensor([[[[ 0.7340, -0.2087,  0.5536]],

         [[ 0.4297,  0.0763, -0.2772]]]])

In [16]:
rearrange(a, "b l n -> b l 1 n") + b

tensor([[[[ 1.4830,  1.3442, -0.2450],
          [ 1.6550, -0.5780,  0.1171],
          [ 1.1162, -0.1522,  0.7572],
          [ 0.5141, -0.8503,  0.4226]],

         [[ 1.1788,  1.6292, -1.0757],
          [ 1.3507, -0.2930, -0.7137],
          [ 0.8119,  0.1327, -0.0736],
          [ 0.2098, -0.5653, -0.4082]]]])

### 语言模型的标准输入格式

`input_ids`  生成过程
1. 文本分词 ：  
   - 使用分词器（如 SentencePiece、BPE、WordPiece 等）将文本分解为 token
   - 例如，"Hello world" 可能被分词为 ["Hello", " world"]
2. ID 映射 ：  
   - 每个 token 被映射到一个唯一的整数 ID
   - 这些 ID 来自分词器的词汇表（vocabulary）
3. 序列填充 ：  
   - 确保所有输入序列长度一致（通过填充或截断）
   - 添加特殊 token（如 [CLS]分类信息、[SEP]分隔句子、[PAD]空白填充、[UNK]词汇表外的词）

`形状： (batch_size: 一个批次中样本数量, seqlen: 序列长度)`

## 设备选择

**`Device`**  ：这是一个自定义的**类型别名**名称  
**`:`** ：类型注解的分隔符  
**`TypeAlias`** ：表示这是一个类型别名  
**`=`** ：赋值操作符  
**`str | torch.device | None`** ：联合类型，表示 Device 可以是 str 、 torch.device 或 None 类型

In [17]:
Device: TypeAlias = str | torch.device | None

检测最佳可用设备: CUDA > MPS (Apple Silicon) > CPU

In [18]:
def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")
get_device()

device(type='cuda')

## Mamba3Config
Mamba3完整模型的配置类

**`@dataclass`** 装饰器，自动生成特殊方法 ：
- **`__init__`** ：构造函数，用于初始化类的实例
- **`__repr__`** ：返回对象的字符串表示
- **`__eq__`** ：用于比较两个对象是否相等
- **`__hash__`** （如果冻结）：用于哈希操作

**属性介绍**  
| 属性名 | 类型 | 默认值 | 含义 |
|-------|------|-------|------|
| `d_model` | `int` | 无 | 模型维度 (D) |
| `n_layer` | `int` | 24 | Mamba-3 层数 (每层 = SSM mixer + SwiGLU MLP) |
| `d_state` | `int` | 128 | SSM 状态维度 (N)。必须为偶数以用于 RoPE 配对。 |
| `expand` | `int` | 2 | 扩展因子 (E)。d_inner = expand * d_model |
| `headdim` | `int` | 64 | 头部维度 (P) |
| `chunk_size` | `int` | 64 | 分块大小（Q） |
| `vocab_size` | `int` | 50277 | 词汇表大小 |
| `pad_vocab_size_multiple` | `int` | 16 | 词汇表大小的填充倍数 |
| `use_mimo` | `bool` | False | 切换 MIMO多输入多输出模式 |
| `mimo_rank` | `int` | 4 | 当use_mimo=True时的秩 (r)理解为不同语义特征数 |

**`__post_init__`** 用来**自动**处理初始化后需要补充的逻辑         
| 属性名 | 计算方法 | 含义 |
|-------|---------|------|
| `d_inner` | `expand * d_model` | 内部维度，由扩展因子乘以模型维度计算得出 |
| `nheads` | `d_inner // headdim` | 注意力头数量，由内部维度除以头部维度计算得出 |
| `d_mlp_inner` | `256 * ((int(2 * (4 * d_model) / 3) + 255) // 256)` | SwiGLU 内部维度，按照 Llama 约定计算（≈ 8/3 * d_model，四舍五入到 256） |
| `vocab_size` | 如果不是 `pad_vocab_size_multiple` 的倍数，则进行调整 | 确保词汇表大小是指定倍数的整数倍 |

In [19]:
@dataclass
class Mamba3Config:
    d_model: int        
    n_layer: int = 24     
    d_state: int = 128  
    expand: int = 2    
    headdim: int = 64   
    chunk_size: int = 64 
    vocab_size: int = 50277
    pad_vocab_size_multiple: int = 16
    use_mimo: bool = False 
    mimo_rank: int = 4    

    def __post_init__(self):
        self.d_inner = self.expand * self.d_model
        assert self.d_inner % self.headdim == 0, "d_inner 必须能被 headdim 整除"
        
        self.nheads = self.d_inner // self.headdim
        assert self.d_state % 2 == 0, "d_state 必须为偶数以用于复数 SSM / RoPE 配对"

        self.d_mlp_inner = 256 * ((int(2 * (4 * self.d_model) / 3) + 255) // 256)

        if self.vocab_size % self.pad_vocab_size_multiple != 0:
            self.vocab_size += (
                self.pad_vocab_size_multiple - self.vocab_size % self.pad_vocab_size_multiple)

In [20]:
a = Mamba3Config(d_model=512, use_mimo=True)
print(f"{a.d_inner=}, {a.nheads=}, {a.d_mlp_inner=}, {a.vocab_size=}")
a

a.d_inner=1024, a.nheads=16, a.d_mlp_inner=1536, a.vocab_size=50288


Mamba3Config(d_model=512, n_layer=24, d_state=128, expand=2, headdim=64, chunk_size=64, vocab_size=50288, pad_vocab_size_multiple=16, use_mimo=True, mimo_rank=4)

## RMSNorm、SiLU和SwiGLU

### RMSNorm 均方根层归一化

**`RMSNorm`** 在 Mamba-3 中有两个主要作用
- 预归一化 ：在每个块（Mamba3 SSM 块和 SwiGLU MLP 块）之前应用，用于稳定训练
- QK-归一化 ：在 B 和 C 投影上应用，类似于 Transformer 中的 QK-Norm，也是稳定训练
- 论文: `https://arxiv.org/abs/1910.07467`

| 属性名 | 类型 | 默认值 | 含义 |
|-------|------|-------|------|
| `d` | `int` | 必填 | 输入特征的维度，即要进行归一化的特征向量的维度 |
| `eps` | `float` | 1e-5 | 一个很小的常数，用于防止除零错误。当输入的平方均值接近零时，添加这个值可以保证数值稳定性 |
| `device` | `Device` | `None` | 指定模型运行的设备（如 "cuda"、"cpu" 或 "mps"），用于将权重参数分配到正确的设备上 |
| `self.weight` | `nn.Parameter` | 全 1 初始化 | 1xd的张量，可学习的缩放参数，在归一化后对特征进行线性缩放，增加模型的表达能力 |

In [21]:
class RMSNorm(nn.Module):
    def __init__(self, d: int, eps: float = 1e-5, device: Device = None):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d, device=device))

    def forward(self, x: Tensor) -> Tensor:
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

**forward解析**
1. `x.pow(2)` 对输入的每个元素求平方
2. `mean(-1, keepdim=True)`  沿最后一个维度计算平均值，保持维度
3. `+ self.eps` 添加小常数，避免除零错误
4. `torch.rsqrt(...)`  计算平方根的倒数，即 1/√(...)
5. `* self.weight` 乘以可学习的权重参数
6. `x * ...`  将输入乘以归一化因子

In [22]:
class RMSNormTest(torch.nn.Module):
    def __init__(self, d: int, eps: float = 1e-5, device=None):
        super().__init__()
        self.eps = eps
        self.weight = torch.nn.Parameter(torch.ones(d, device=device))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        print(f"输入 x: {x}")
        x_squared = x.pow(2)
        print(f"平方后: {x_squared}")
        mean_squared = x_squared.mean(-1, keepdim=True)
        print(f"平方均值: {mean_squared}")
        mean_squared_eps = mean_squared + self.eps
        print(f"加 eps 后: {mean_squared_eps}")
        rsqrt = torch.rsqrt(mean_squared_eps)
        print(f"平方根倒数: {rsqrt}")
        normalized = x * rsqrt
        print(f"归一化后: {normalized}")
        output = normalized * self.weight
        print(f"应用权重后: {output}")
        return output
        
rms_norm = RMSNormTest(d=3)
x = torch.tensor([[1.0, 2.0, 3.0]])
print(f"{x.shape=}")
print("=============== RMSNorm 计算过程 ===============")
output = rms_norm(x)
print("=================== 最终输出 ===================")
print(f"输出: {output}")

x.shape=torch.Size([1, 3])
=============== RMSNorm 计算过程 ===============
输入 x: tensor([[1., 2., 3.]])
平方后: tensor([[1., 4., 9.]])
平方均值: tensor([[4.6667]])
加 eps 后: tensor([[4.6667]])
平方根倒数: tensor([[0.4629]])
归一化后: tensor([[0.4629, 0.9258, 1.3887]])
应用权重后: tensor([[0.4629, 0.9258, 1.3887]], grad_fn=<MulBackward0>)
=================== 最终输出 ===================
输出: tensor([[0.4629, 0.9258, 1.3887]], grad_fn=<MulBackward0>)


### SiLU : Sigmoid 线性单元 (SiLU) / Swish 激活函数

- `x: Tensor` ：表示参数 x 的类型应该是 Tensor
- `-> Tensor` ：表示函数的返回值类型应该是 Tensor

In [23]:
def silu(x: Tensor) -> Tensor:
    # 与 MPS (Apple Silicon) 兼容的定义
    return x * F.sigmoid(x)

In [24]:
a = torch.randn(10)
print(f"{a=}")
silu(a)

a=tensor([ 0.4670,  0.7919, -0.9966,  0.3335, -0.8829, -0.1365, -2.2270,  0.0410,
         0.8867,  0.5049])


tensor([ 0.2871,  0.5450, -0.2687,  0.1943, -0.2583, -0.0636, -0.2168,  0.0209,
         0.6279,  0.3148])

### SwiGLU Swish门控线性单元

| 属性名 | 类型 | 描述 | 作用 |
|-------|------|------|------|
| `d_model` | `int` | 输入和输出的特征维度 | 定义网络的输入和输出大小 |
| `d_inner` | `int` | 内部隐藏层的维度 | 控制网络的表达能力 |
| `device` | `Device` | 模型运行的设备 | 将模型参数分配到正确的设备上 |
| `w_gate` | `nn.Linear` | 门控线性层 | 计算门控信号，形状: (d_model, d_inner) |
| `w_up` | `nn.Linear` | 上投影线性层 | 计算上投影信号，形状: (d_model, d_inner) |
| `w_down` | `nn.Linear` | 下投影线性层 | 将内部表示映射回输出维度，形状: (d_inner, d_model) |

**forward解析**
1. **门控计算**：`gate = silu(w_gate(x))`
2. **上投影**：`up = w_up(x)`
3. **门控操作**：`gate * up`
4. **下投影**：`w_down(gate * up)`

SwiGLU(x) = W_down(SiLU(W_gate(x)) ⊙ W_up(x))  
形状: (batch, seqlen, d_model) → (batch, seqlen, d_model)

In [25]:
class SwiGLU(nn.Module):
    def __init__(self, d_model: int, d_inner: int, device: Device = None):
        super().__init__()
        self.w_gate = nn.Linear(d_model, d_inner, bias=False, device=device)
        self.w_up = nn.Linear(d_model, d_inner, bias=False, device=device)
        self.w_down = nn.Linear(d_inner, d_model, bias=False, device=device)

    def forward(self, x: Tensor) -> Tensor:
        return self.w_down(silu(self.w_gate(x)) * self.w_up(x))

In [26]:
print("===== SwiGLU 测试 =====")
# 配置参数
d_model = 4  
d_inner = 8  
batch_size = 1
seq_len = 2

# 创建 SwiGLU 实例
swiglu = SwiGLU(d_model, d_inner)

# 打印模型结构
print(f"SwiGLU 配置: d_model={d_model}, d_inner={d_inner}")
print(f"w_gate 权重形状: {swiglu.w_gate.weight.shape}")
print(f"w_up 权重形状: {swiglu.w_up.weight.shape}")
print(f"w_down 权重形状: {swiglu.w_down.weight.shape}")

# 创建测试输入 (batch, seq_len, d_model)
x = torch.tensor([[[1.0, 2.0, 3.0, 4.0],
                   [5.0, 6.0, 7.0, 8.0]]], dtype=torch.float32)
print(f"\n输入形状: {x.shape}")
print(f"输入:\n{x}")

# 前向传播
output = swiglu(x)
print(f"\n输出形状: {output.shape}")
print(f"输出:\n{output}")

===== SwiGLU 测试 =====
SwiGLU 配置: d_model=4, d_inner=8
w_gate 权重形状: torch.Size([8, 4])
w_up 权重形状: torch.Size([8, 4])
w_down 权重形状: torch.Size([4, 8])

输入形状: torch.Size([1, 2, 4])
输入:
tensor([[[1., 2., 3., 4.],
         [5., 6., 7., 8.]]])

输出形状: torch.Size([1, 2, 4])
输出:
tensor([[[-1.4489, -0.9587, -0.5285, -1.3623],
         [-5.6391, -3.0060, -1.6070, -5.9522]]], grad_fn=<UnsafeViewBackward0>)


## 推理缓存 InferenceCache

- 状态保持 ：存储前一个时间步的状态，避免重复计算
- 高效推理 ：使得每个时间步的计算复杂度为常数 O(1)，训练时是 O(L)

In [27]:
class InferenceCache(NamedTuple):
    ssm_state: Tensor   # (batch, nheads, headdim, d_state) — SSM 隐藏状态 h_t
    prev_Bx: Tensor     # (batch, nheads, headdim, d_state) — 用于 β 项的 B̄_{t-1} ⊗ x_{t-1}
    cum_angle: Tensor   # (batch, nheads, d_state // 2)  — 累积 RoPE 角度 Σ Δ_i * θ_i

    @staticmethod
    def alloc(batch_size: int, args: Mamba3Config, device: Device = None):
        return InferenceCache(
            ssm_state=torch.zeros(
                batch_size, args.nheads, args.headdim, args.d_state, device=device
            ),
            prev_Bx=torch.zeros(
                batch_size, args.nheads, args.headdim, args.d_state, device=device
            ),
            cum_angle=torch.zeros(
                batch_size, 1, args.d_state // 2, device=device
            ),
        )

## 数据依赖RoPE

应用带有数据依赖角度的旋转位置嵌入。
- 复数 SSM 的块对角旋转矩阵 R_t 通过 "RoPE 技巧" 被吸收到 B 和 C 中。
- 累积旋转角度是数据依赖的 (从输入投影)，与使用固定频率调度的标准 RoPE 不同。
- x: (..., d_state) 要旋转的 B 或 C 投影, angles: (..., d_state // 2) 累积旋转角度

In [38]:
def apply_rope(x: Tensor, angles: Tensor) -> Tensor:
    # 分割为对: 偶维度 (2j) 和奇维度 (2j+1)
    x1 = x[..., 0::2]  # (..., d_state // 2) — 偶索引
    x2 = x[..., 1::2]  # (..., d_state // 2) — 奇索引

    # 计算余弦和正弦值
    cos_a = torch.cos(angles)
    sin_a = torch.sin(angles)

    # 每对的 2D 旋转: R(θ) = [[cos θ, −sin θ], [sin θ, cos θ]]
    x_rot_even = cos_a * x1 - sin_a * x2   
    x_rot_odd = sin_a * x1 + cos_a * x2

    # 重新交织偶和奇
    # 沿最后一个维度堆叠然后将最后两个维度展平: (..., d_state//2, 2) → (..., d_state)
    return torch.stack([x_rot_even, x_rot_odd], dim=-1).flatten(-2)

## segsum SSM 中的稳定段和计算 （衰减率）

1-半可分矩阵生成 ： exp(segsum(A)) 产生一个 1-半可分矩阵（下三角衰减掩码）   
输入：`x[b, h, c, l]` 表示第 b 个批次、第 h 个头部、第 c 个块、第 l 个时间步的累积衰减率   
输出：`x_segsum[b, h, c, i, j]` 表示从时间步 j+1 到 i 的衰减率之和（当 i ≥ j 时）

In [39]:
def segsum(x: Tensor, device: Device = None) -> Tensor:
    """稳定的段和计算。
    exp(segsum(A)) 产生一个 1-半可分矩阵 (下三角衰减掩码)，
    这相当于标量 SSM 的累积衰减产物。
    segsum(x)[..., i, j] = Σ_{k=j+1}^{i} x[..., k]   对于 i ≥ j，否则 −∞
    来源: Mamba-2 (Dao & Gu, 2024)
    """
    # 获取时间维度
    T = x.size(-1)
    
    # 扩展维度 ，将输入 最后一维 扩展为 二维T×T 矩阵
    x = repeat(x, "... d -> ... d e", e=T)
    
    # 创建下三角掩码 下三角true，其他false
    # diagonal=0 （默认）：包含主对角线，保留主对角线及以下的元素
    # diagonal=1 ：保留主对角线上方的一条对角线及以下的元素
    # diagonal=-1 ：只保留主对角线下方的一条对角线及以下的元素，即 严格下三角矩阵 （不包含主对角线）
    mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=device), diagonal=-1)
    
    # 应用掩码，将非下三角部分设为 0（对角线也是0）其他数值不变
    # ~mask 表示 mask 取反， 即 true的地方设置为0
    x = x.masked_fill(~mask, 0)
    
    #累积求和，沿倒数第二维度累积求和（计算第一个位置（时间步）到当前位置（时间步）的累计）
    x_segsum = torch.cumsum(x, dim=-2)
    
    # 再次应用掩码，这次包括对角线 ： 上三角false，其他true
    mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=device), diagonal=0)
    
    # 设置上三角为 -inf
    x_segsum = x_segsum.masked_fill(~mask, -torch.inf)
    
    return x_segsum

In [40]:
x = torch.tensor([[-0.1, -0.2, -0.3]])  # shape: (1, 3)
segsum(x)

tensor([[[ 0.0000,    -inf,    -inf],
         [-0.2000,  0.0000,    -inf],
         [-0.5000, -0.3000,  0.0000]]])

**衰减系数**

In [41]:
torch.exp(segsum(x))

tensor([[[1.0000, 0.0000, 0.0000],
         [0.8187, 1.0000, 0.0000],
         [0.6065, 0.7408, 1.0000]]])

`0.6065`表示从 `时间步0` 到 `时间步2` 的状态只剩下`60.65%`

## SSD 结构化状态空间对偶性

### SISO

Arguments
- `x: (batch, seqlen, nheads, headdim)`  — pre-scaled SSM input
- `A: (batch, seqlen, nheads)`          ————— log-decay rates (Δ * A, already negative)
- `B: (batch, seqlen, nheads, d_state)`  —— input projection (per-head in Mamba-3)
- `C: (batch, seqlen, nheads, d_state)`  —— output projection (per-head in Mamba-3)
- `chunk_size: int`                       —————————— partition size Q

Return
- `y: (batch, seqlen, nheads, headdim)`
- `final_state: (batch, nheads, headdim, d_state)`

In [42]:
# ssd(x * gamma.unsqueeze(-1), dA, B, C, args.chunk_size, device=self.device)
def ssd(x, A, B, C, chunk_size, initial_states=None, device: Device = None):
    """
    Return
        y: (batch, seqlen, n_heads, d_head)
        final_state: (batch, n_heads, d_head, d_state)
    """
    
    # 要求序列长度整除分块大小 seqlen % chunk_size = 0
    assert x.shape[1] % chunk_size == 0, (
        f"seqlen ({x.shape[1]}) must be divisible by chunk_size ({chunk_size})"
    )

    # 将输入张量 x 、衰减率 A 、输入投影 B 和输出投影 C 重排为分块形式
    # (batch, seqlen, ...) -> (batch, nchunks, chunk_size, ...)
    x, A, B, C = [
        rearrange(m, "b (c l) ... -> b c l ...", l=chunk_size) for m in (x, A, B, C)
    ]

    # 调整 A 的维度顺序，方便后续计算
    A = rearrange(A, "b c l h -> b h c l")
    # 在分块大小维度上（其实就是对每个时间步的衰减）计算第一个位置到当前位置的累计
    A_cumsum = torch.cumsum(A, dim=-1)

    # ==========================块内计算，对角块==========================
    # L 为衰减掩码矩阵   (batch, nheads, nchunks, chunk_size, chunk_size)
    L = torch.exp(segsum(A, device=device))   
    
    # 先计算 B和 x的外积 (batch, nchunks, chunk_size, n_heads, d_state, headdim)
    # 1. Bx = torch.einsum("bcshn, bcshp -> bcshnp", B, x)
    
    # 应用衰减并对源位置 s 求和 (batch, nchunks, n_heads, chunk_size, d_state, headdim)
    # L_reshaped = rearrange(L, "b h c l s -> b c h l s")
    # 2. L_Bx = torch.einsum("bchls, bcshnp -> bchlnp", L_reshaped, Bx)

    # 应用输出投影并对状态维度 n 求和
    # L_Bx_reshaped = rearrange(L_Bx, "b c h l n p -> b c l h n p")
    # Y_diag = torch.einsum("bclhn, bclhnp -> bclhp", C, L_Bx_reshaped)  = CL_Bx
    Y_diag = torch.einsum("bclhn, bcshn, bhcls, bcshp -> bclhp", C, B, L, x)
    #=====================================================================

    # ==========================计算每个块的状态===========================
    # A_cumsum: (batch, nheads, nchunks, chunk_size)
    # 最后一个时间步的数值 - 每个时间步的数值    得到每个时间步对于最后一个时间步的累计衰减和
    # -1: 表示把最后一个数值作为一个列表，作减法时自动会填充成A_cumsum同样长度
    # 此处对每个chunk内计算每个输入到块末尾的累计衰减（遗忘程度）
    decay_states = torch.exp(A_cumsum[:, :, :, -1:] - A_cumsum)
    
    # 计算 decay_states 和 x 的乘积  (batch, nchunks, nheads, chunk_size, headdim)
    # decay_states_reshaped = rearrange(decay_states, "b h c l -> b c h l")
    # decay_x = torch.einsum("bchl, bclhp -> bchlp", decay_states_reshaped, x)

    # 计算 B 和 decay_x 的乘积，并对时间步 l求和  (batch, nchunks, nheads, headdim, d_state)
    # B_reshaped = rearrange(B, "b c l h n -> b c h l n")
    # states = torch.einsum("bchln, bchlp -> bchpn", B_reshaped, decay_x)   = L_Bx
    states = torch.einsum("bclhn, bhcl, bclhp -> bchpn", B, decay_states, x)
    #====================================================================

    # =========================块间 SSM 递归 (A项) ======================
    # 初始化初始状态
    if initial_states is None:
        initial_states = torch.zeros_like(states[:, :1])
        
    # 将初始状态与所有块的状态拼接  (batch, nchunks+1, nheads, headdim, d_state)
    states = torch.cat([initial_states, states], dim=1)
    
    # 计算块间的衰减因子
    # A_cumsum[:, :, :, -1] : (batch, n_heads, n_chunks)
    # F.pad(..., (1, 0)) → 形状：(batch, n_heads, n_chunks+1), 最后一维第一位置添加一个 0
    # segsum: (batch, nheads, nchunks+1, nchunks+1)
    # 此处对每个chunk计算每个chunk间的累计衰减（遗忘程度）
    decay_chunk = torch.exp(
        segsum(F.pad(A_cumsum[:, :, :, -1], (1, 0)), device=device)
    )
    
    # 计算块间传播后的新状态
    # decay_chunk 形状： (batch, nheads, nchunks+1, nchunks+1)
    # states      形状： (batch, nchunks+1, nheads, headdim, d_state)
    # new_states  形状： (batch, nchunks+1, nheads, headdim, d_state)
    new_states = torch.einsum("bhzc, bchpn -> bzhpn", decay_chunk, states)

    # 分离出每个块的状态和最终状态
    # :-1 表示不包括最后一个    -1 表示最后一个
    # states :      (batch, nchunks, nheads, headdim, d_state)
    # final_state : (batch,          nheads, headdim, d_state) 
    states, final_state = new_states[:, :-1], new_states[:, -1]
    # ====================================================================

    # ==========================状态到输出的转换============================
    # 计算状态到输出的衰减因子
    # 此处为从 当前块的开始 到 当前块内的每个时间步   之前是最后一个 - 任意，现在是任意 - 第一个
    state_decay_out = torch.exp(A_cumsum)
    
    # 将每个块的状态转换为输出 C L_Bx 
    # 计算 states 和 state_decay_out 的乘积（对时间步 l 应用衰减） 
    # state_decay_out_reshaped = rearrange(state_decay_out, "b h c l -> b c h l")
    # decayed_states = torch.einsum("bchpn, bchl -> bchpln", states, state_decay_out_reshaped)
    
    # 计算 C 和 decayed_states 的乘积，并对状态维度 n 求和
    # C_reshaped = rearrange(C, "b c l h n -> b c h l n")
    # decayed_states_reshaped = rearrange(decayed_states, "b c h l p n -> b c h l n p")
    # Y_off = torch.einsum("bchln, bchlnp -> bchlp", C_reshaped, decayed_states_reshaped)
    # Y_off = rearrange(Y_off, "b c h l p -> b c l h p")
    Y_off = torch.einsum("bclhn, bchpn, bhcl -> bclhp", C, states, state_decay_out)
    # ====================================================================

    # ==============================合并输出===============================
    # 将块内输出 Y_diag 和块间输出 Y_off 相加，重排回原始形状
    # Y: (batch, seqlen, nheads, headdim)
    Y = rearrange(Y_diag + Y_off, "b c l h p -> b (c l) h p")
    # ====================================================================

    return Y, final_state

### MIMO

Arguments
- `x: (batch, seqlen, nheads, headdim, mimo_rank)`  — pre-scaled SSM input
- `A: (batch, seqlen, nheads)`          ——————————— log-decay rates (Δ * A, already negative)
- `B: (batch, seqlen, nheads, d_state, mimo_rank)`  —— input projection (per-head in Mamba-3)
- `C: (batch, seqlen, nheads, d_state, mimo_rank)`  —— output projection (per-head in Mamba-3)
- `chunk_size: int`                       —————————— partition size Q

Return
- `y: (batch, seqlen, nheads, headdim, mimo_rank)`
- `final_state: (batch, nheads, headdim, d_state)`

In [43]:
def ssd_mimo(x, A, B, C, chunk_size, initial_states=None, device: Device = None):
    """结构化状态空间对偶性 — MIMO 变体.
        MIMO 公式通过将基于外积的状态更新 (b ⊗ x) 替换为基于矩阵乘积的更新 (B @ X^T) 来泛化 SISO。
        MIMO 秩 R 与序列维度正交，因此 Mamba-2 分块 SSD 算法适用于带有修改的 einsum，这些 einsum 在 R 上收缩。
        状态形状与 SISO 相同: (batch, n_heads, d_head, d_state)。
    """

    #============================以下同 SISO =============================
    assert x.shape[1] % chunk_size == 0, (
        f"seqlen ({x.shape[1]}) must be divisible by chunk_size ({chunk_size})"
    )
    x, A, B, C = [
        rearrange(m, "b (c l) ... -> b c l ...", l=chunk_size) for m in (x, A, B, C)
    ]
    A = rearrange(A, "b c l h -> b h c l")
    A_cumsum = torch.cumsum(A, dim=-1)

    # ==========================块内计算，对角块==========================
    # L 为衰减掩码矩阵   (batch, nheads, nchunks, chunk_size, chunk_size)
    L = torch.exp(segsum(A, device=device))
    # 对比SISO的 Y_diag = torch.einsum("bclhn, bcshn, bhcls, bcshp -> bclhp", C, B, L, x)
    # bcshnr、bcshpr -> bcshnp ?
    # bhcls、bcshnp -> bchlnp ?
    # bclhnq、bclhnp -> bclhpq ?
    Y_diag = torch.einsum("bclhnq, bcshnr, bhcls, bcshpr -> bclhpq", C, B, L, x)

    # ==========================计算每个块的状态===========================
    decay_states = torch.exp(A_cumsum[:, :, :, -1:] - A_cumsum)
    # 对比SISO的 states = torch.einsum("bclhn, bhcl, bclhp -> bchpn", B, decay_states, x)
    # bhcl、bclhpr -> bclhpr ?
    # bclhnr、bclhpr -> bchpn ?
    states = torch.einsum("bclhnr, bhcl, bclhpr -> bchpn", B, decay_states, x)

    # =========================块间 SSM 递归 同 SISO ======================
    if initial_states is None:
        initial_states = torch.zeros_like(states[:, :1])
    states = torch.cat([initial_states, states], dim=1)
    decay_chunk = torch.exp(
        segsum(F.pad(A_cumsum[:, :, :, -1], (1, 0)), device=device)
    )
    new_states = torch.einsum("bhzc, bchpn -> bzhpn", decay_chunk, states)
    states, final_state = new_states[:, :-1], new_states[:, -1]

    # ==========================状态到输出的转换============================
    state_decay_out = torch.exp(A_cumsum)
    # 对比SISO的 Y_off = torch.einsum("bclhn, bchpn, bhcl -> bclhp", C, states, state_decay_out)
    # bchpn、bhcl -> bclhnp ?
    # bclhnr、bclhnp -> bclhpr ?
    Y_off = torch.einsum("bclhnr, bchpn, bhcl -> bclhpr", C, states, state_decay_out)

    # ==============================合并输出===============================
    # 对比SISO的 Y = rearrange(Y_diag + Y_off, "b c l h p -> b (c l) h p")
    Y = rearrange(Y_diag + Y_off, "b c l h p r -> b (c l) h p r")

    return Y, final_state

## Mamba3块

**Mamba3块初始化**
| 属性名 | 类型 | 默认值 | 描述 |
|-------|------|-------|------|
| `args` | `Mamba3Config` | 必填 | 模型配置对象，包含各种超参数 |
| `device` | `Device` | `None` | 模型运行的设备 |
| `bc_dim` | `int` | 计算得出 | 投影维度，SISO 时为 d_state，MIMO 时为 d_state * mimo_rank |
| `in_proj` | `nn.Linear` | 线性层 | 输入投影层，将 d_model 投影到 z + x + B + C + dt + λ + θ |
| `A_log` | `nn.Parameter` | 空可学习参数 | 负对角线的对数，exp 后始终为负 |
| `D` | `nn.Parameter` | 空可学习参数 | 跳跃连接系数，每个头部一个 |
| `dt_bias` | `nn.Parameter` | 空可学习参数 | softplus 前添加到 dt 的偏置 |
| `B_norm` | `RMSNorm` | 归一化层 | B 投影上的 QK-归一化 |
| `C_norm` | `RMSNorm` | 归一化层 | C 投影上的 QK-归一化 |
| `B_bias` | `nn.Parameter` | 全1初始化可学习参数 | B 投影的偏置，头部特定的、通道方向的 |
| `C_bias` | `nn.Parameter` | 全1初始化可学习参数 | C 投影的偏置，头部特定的、通道方向的 |
| `mimo_x_proj` | `nn.Parameter` | 全1初始化可学习参数 | MIMO 秩扩展参数，仅在 use_mimo=True 时存在 |
| `mimo_z_proj` | `nn.Parameter` | 全1初始化可学习参数 | MIMO 秩扩展参数，仅在 use_mimo=True 时存在 |
| `mimo_down` | `nn.Parameter` | 全1/R初始化可学习参数 | MIMO 秩下投影参数，仅在 use_mimo=True 时存在 |
| `out_proj` | `nn.Linear` | 线性层 | 输出投影层，将内部表示映射回 d_model 维度 |

**SSD（SSM）参数**
| 参数名 | 形状 | 描述 | 作用 |
|-------|------|------|------|
| `z` | `(batch, seqlen, d_inner)` | 输出的门控 | 使用 SiLU 激活，控制信息流动 |
| `x` | `(batch, seqlen, d_inner)` | SSM 输入值 | 作为 SSM 的输入信号 |
| `B` | `(batch, seqlen, bc_dim)` | SSM 输入投影 | 将输入映射到状态空间，SISO 时为 d_state，MIMO 时为 d_state*R |
| `C` | `(batch, seqlen, bc_dim)` | SSM 输出投影 | 将状态空间映射回输出空间 |
| `dt` | `(batch, seqlen, nheads)` | 步长 Δ | 每个头部一个，控制状态更新的步长 |
| `λ` | `(batch, seqlen, nheads)` | 梯形插值参数 | 每个头部一个，控制梯形离散化的插值权重 |
| `θ` | `(batch, seqlen, d_state // 2)` | 数据依赖 RoPE 的旋转角度 | 用于计算复数 SSM 的旋转角度 |

- **z 和 x**：分别用于门控和 SSM 输入
- **B 和 C**：分别用于输入和输出投影，是 SSM 的关键参数
- **dt 和 λ**：控制梯形离散化的参数，影响状态更新的方式
- **θ**：用于数据依赖的 RoPE 计算，实现复数 SSM 的状态跟踪

In [44]:
class Mamba3(nn.Module):
    def __init__(self, args: Mamba3Config, device: Device = None):
        super().__init__()
        self.args = args
        self.device = device
        
        self.bc_dim = args.d_state * args.mimo_rank if args.use_mimo else args.d_state
        
        d_in_proj = (
            2 * args.d_inner     # z + x
            + 2 * self.bc_dim    # B + C (MIMO 时由 mimo_rank 扩展)
            + 2 * args.nheads    # dt + λ
            + args.d_state // 2  # θ
        )
        
        self.in_proj = nn.Linear(args.d_model, d_in_proj, bias=False, device=device)

        # ── SSM 参数 ──
        self.A_log = nn.Parameter(torch.empty(args.nheads, device=device))
        self.D = nn.Parameter(torch.empty(args.nheads, device=device))
        self.dt_bias = nn.Parameter(torch.empty(args.nheads, device=device))

        # ── B, C 上的 QK-归一化层 ──
        self.B_norm = RMSNorm(self.bc_dim, device=device)
        self.C_norm = RMSNorm(self.bc_dim, device=device)

        # ── BC 偏置  ──
        if args.use_mimo:
            R = args.mimo_rank  #秩
            
            # BC偏置形状（nheads，d_state，R）
            self.B_bias = nn.Parameter(torch.ones(args.nheads, args.d_state, R, device=device))
            self.C_bias = nn.Parameter(torch.ones(args.nheads, args.d_state, R, device=device))
            
            # MIMO 秩扩展 （nheads，headdim，R）
            self.mimo_x_proj = nn.Parameter(torch.ones(args.nheads, args.headdim, R, device=device))
            self.mimo_z_proj = nn.Parameter(torch.ones(args.nheads, args.headdim, R, device=device))
            
            # MIMO 秩下投影: (P, R) → (P) 通过学习的加权和 （头数，头部维度，秩）
            self.mimo_down = nn.Parameter(torch.ones(args.nheads, args.headdim, R, device=device) / R)
        else:
            # SISO: 偏置为 (nheads，d_state)
            self.B_bias = nn.Parameter(torch.ones(args.nheads, args.d_state, device=device))
            self.C_bias = nn.Parameter(torch.ones(args.nheads, args.d_state, device=device))

        # ── 输出投影 ──
        self.out_proj = nn.Linear(args.d_inner, args.d_model, bias=False, device=device)

**forward**
1. 升维：(batch, seqlen, d_model) -> (batch, seqlen, d_in_proj)
2. 分割输入投影
    - z: (batch, seqlen, d_inner)
    - x: (batch, seqlen, d_inner)
    - B: (batch, seqlen, bc_dim)
    - C: (batch, seqlen, bc_dim)
    - dt: (batch, seqlen, nheads)
    - lam: (batch, seqlen, nheads)
    - theta: (batch, seqlen, d_state//2)
3. 初始化 dt 和 lam ，限定范围
4. BC归一化，稳定训练
5. 数据依赖
6. 准备ssd参数：alpha：状态衰减系数，beta：前一个输入对当前状态影响，gamma：当前输入对当前状态影响
7. 调整输入x的形状适配ssd
8. SSM

**SSM**
1. 添加头部特定的 BC 偏置(nheads, d_state, R)
2. 对 B 和 C 应用累积旋转
3. 梯形 SSD (双 SSD 分解，MIMO 变体)
4. 求和两个贡献
5. 秩-R 空间中的跳跃连接
6. 秩-R 空间中的门控
7. 输出投影
8. 构建推理缓存

In [47]:
def Mamba3forward(self, u: Tensor, h: InferenceCache | None = None):
    # 如果提供了推理缓存，则使用推理步骤
    if h is not None:
        return self.step(u, h)

    # 获取输入张量的形状
    batch, seqlen, _ = u.shape
    args = self.args

    # 初始化状态转换全 -1矩阵 A,形状: (nheads,)
    A = -torch.exp(self.A_log) 

    #1. 升维 (batch, seqlen, d_model) -> (batch, seqlen, d_in_proj)
    proj = self.in_proj(u)  
    
    # 2. 分割输入投影的输出为多个分支
    z, x, B, C, dt, lam, theta = torch.split(
        proj,
        [
            args.d_inner,        # z: 门控
            args.d_inner,        # x: SSM 输入
            self.bc_dim,         # B: 输入投影 (SISO 为 d_state, MIMO 为 d_state*R)
            self.bc_dim,         # C: 输出投影
            args.nheads,         # dt: 每个头部的步长
            args.nheads,         # λ: 每个头部的梯形参数
            args.d_state // 2,   # θ: 旋转角度
        ],
        dim=-1,)
    
    # 3. softplus(x) = ln(1 + exp(x)) 保证步长 dt 必须为正
    dt = F.softplus(dt + self.dt_bias) 
    # sigmoid(x) = 1 / (1 + exp(-x)) lam 表示当前输入的权重，1-lam 表示前一个输入的权重
    lam = torch.sigmoid(lam) 

    # 4. B, C 上的 QK-归一化 (仅 RMSNorm)
    B = self.B_norm(B)  
    C = self.C_norm(C) 

    # 5.数据依赖的 RoPE 
    # 元素乘结果 形状: (batch, seqlen, nheads, d_state // 2)
    raw_angles = (
        dt.unsqueeze(-1)                         # (b, l, nheads) -> (b, l, nheads, 1) 广播后-> (b, l, nheads, d_state//2)
        * rearrange(theta, "b l n -> b l 1 n")   # (b, l, d_state//2) -> (b, l, 1, d_state//2) 广播后-> (b, l, nheads, d_state//2)
    )  

    #在 seqlen维度 计算第一个位置到当前位置的累计
    cum_angles = -torch.cumsum(raw_angles, dim=1)  # 形状: (batch, seqlen, nheads, d_state // 2)

    # 6.梯形系数 
    #   α_t = exp(Δ_t * A_t)                     — 衰减
    #   β_t = (1 − λ_t) * Δ_t * exp(Δ_t * A_t)   — 左端点 (前一个输入)
    #   γ_t = λ_t * Δ_t                          — 右端点 (当前输入)
    dA = dt * rearrange(A, "h -> 1 1 h")         # (batch, seqlen, nheads)
    alpha = torch.exp(dA)                        # (batch, seqlen, nheads)
    beta = (1 - lam) * dt * alpha                # (batch, seqlen, nheads)
    gamma = lam * dt                             # (batch, seqlen, nheads)

    # 7.x: (batch, seqlen, d_inner) -> (batch, seqlen, nheads, headdim) 
    x = rearrange(x, "b l (h p) -> b l h p", p=args.headdim) 

    if args.use_mimo:
        # ── MIMO 路径 ──
        # 获取秩的值
        R = args.mimo_rank

        # 1.添加头部特定的 BC 偏置(nheads, d_state, R)
        # B, C: (b, l, d_state*R) -> (b, l, d_state, R) 
        B = rearrange(B, "b l (n r) -> b l n r", r=R)  
        C = rearrange(C, "b l (n r) -> b l n r", r=R) 

        # B、C: (batch, seqlen, d_state, R) -> (batch, seqlen, 1, d_state, R) -> (b, l, nheads, d_state, R)
        # B_bias、C_bias: (nheads, d_state, R) -> (1, 1, nheads, d_state,R) -> (b, l, nheads, d_state, R)
        B = rearrange(B, "b l n r -> b l 1 n r") + self.B_bias 
        C = rearrange(C, "b l n r -> b l 1 n r") + self.C_bias  

        # 2.对 B 和 C 应用累积旋转
        B = rearrange(B, "b l h n r -> b l h r n")  # 形状: (batch, seqlen, nheads, R, d_state)
        C = rearrange(C, "b l h n r -> b l h r n")  # 形状: (batch, seqlen, nheads, R, d_state)

        # cum_angles: (batch, seqlen, nheads, d_state//2) -> (batch, seqlen, nheads, 1, d_state//2)
        B = apply_rope(B, cum_angles.unsqueeze(3))  # 广播到秩维度
        C = apply_rope(C, cum_angles.unsqueeze(3))
        
        B = rearrange(B, "b l h r n -> b l h n r")  # (b, l, nheads, d_state, R)
        C = rearrange(C, "b l h r n -> b l h n r")  # (b, l, nheads, d_state, R)

        # ── 将 x 扩展到秩 R : X_t = X'_t ⊙ W_X ──
        # x: (batch, seqlen, nheads, headdim) -> (batch, seqlen, nheads, headdim, 1)
        # mimo_x_proj: (nheads, headdim, R) -> (1, 1, nheads, headdim, R)
        # x_mimo: (batch, seqlen, nheads, headdim, R)
        x_mimo = x.unsqueeze(-1) * self.mimo_x_proj 

        # 3.梯形 SSD (双 SSD 分解，MIMO 变体) 
        # y_gamma: (batch, seqlen, nheads, headdim, R)
        # state_gamma: (batch, nheads, headdim, d_state)
        y_gamma, state_gamma = ssd_mimo(
            x_mimo * rearrange(gamma, "b l h -> b l h 1 1"),  # 按 γ 缩放
            dA, B, C, args.chunk_size, device=self.device,
        )  

        # 为 β 项在完整序列级别预移 B 和 x
        B_prev = F.pad(B[:, :-1], (0, 0, 0, 0, 0, 0, 1, 0))            # (b, l, h, n, R)
        x_mimo_prev = F.pad(x_mimo[:, :-1], (0, 0, 0, 0, 0, 0, 1, 0))  # (b, l, h, p, R)

        # y_beta: (batch, seqlen, nheads, headdim, R)
        # state_beta: (batch, nheads, headdim, d_state)
        y_beta, state_beta = ssd_mimo(
            x_mimo_prev * rearrange(beta, "b l h -> b l h 1 1"),
            dA, B_prev, C, args.chunk_size, device=self.device,
        )  

        # 4.求和两个贡献
        y = y_gamma + y_beta                              # 形状: (batch, seqlen, nheads, headdim, R)
        ssm_state = state_gamma + state_beta              # 形状: (batch, nheads, headdim, d_state)

        # 5.秩-R 空间中的跳跃连接
        y = y + (x * self.D.unsqueeze(-1)).unsqueeze(-1)  # 广播到 R

        # 6.秩-R 空间中的门控
        z_heads = rearrange(z, "b l (h p) -> b l h p", p=args.headdim)  # 形状: (batch, seqlen, nheads, headdim)
        z_mimo = z_heads.unsqueeze(-1) * self.mimo_z_proj  # 形状: (batch, seqlen, nheads, headdim, R)
        y = y * silu(z_mimo)

        # 7.输出投影秩: (b, l, h, p, R) → (b, l, h, p) ──
        y = (y * self.mimo_down).sum(dim=-1)               # 形状: (batch, seqlen, nheads, headdim)
        y = rearrange(y, "b l h p -> b l (h p)")           # 形状: (batch, seqlen, d_inner)
        y = self.out_proj(y)                               # 形状: (batch, seqlen, d_model)

        # 8.构建推理缓存
        # 收缩秩 R 后的 B⊗x → 与 SISO 状态相同的形状
        last_Bx = torch.einsum(                            # 形状: (batch, nheads, headdim, d_state)    
            "bhnr, bhpr -> bhpn",
            B[:, -1], x_mimo[:, -1],
        )   

    else:
        # ── SISO 路径 ──
        # 1.添加头部特定的 BC 偏置(nheads, d_state)
        # B、C: (batch, seqlen, d_state) -> (batch, seqlen, 1, d_state) -> (b, l, nheads, d_state)
        # B_bias、C_bias: (nheads, d_state) -> (1, 1, nheads, d_state) -> (b, l, nheads, d_state)
        B = rearrange(B, "b l n -> b l 1 n") + self.B_bias  
        C = rearrange(C, "b l n -> b l 1 n") + self.C_bias  

        # 2.对 B 和 C 应用累积旋转
        B = apply_rope(B, cum_angles)  # 形状: (batch, seqlen, nheads, d_state)
        C = apply_rope(C, cum_angles)  # 形状: (batch, seqlen, nheads, d_state)

        # 3.梯形 SSD (双 SSD 分解)
        # y_gamma: (batch, seqlen, nheads, headdim)
        # state_gamma: (batch, nheads, headdim, d_state)
        y_gamma, state_gamma = ssd(
            x * gamma.unsqueeze(-1),        # 形状保持不变 (batch, seqlen, nheads, headdim)
            dA,                             # (batch, seqlen, nheads)
            B,                              # 形状: (batch, seqlen, nheads, d_state)
            C,                              # 形状: (batch, seqlen, nheads, d_state)
            args.chunk_size, device=self.device,
        )  

        # 为 β 项预移 B 和 x
        B_prev = F.pad(B[:, :-1], (0, 0, 0, 0, 1, 0))  # 右移
        x_prev = F.pad(x[:, :-1], (0, 0, 0, 0, 1, 0))

        # y_beta: (batch, seqlen, nheads, headdim)
        # state_beta: (batch, nheads, headdim, d_state)
        y_beta, state_beta = ssd(
            x_prev * beta.unsqueeze(-1), dA, B_prev, C,
            args.chunk_size, device=self.device,
        )  

        # 4.求和两个贡献
        y = y_gamma + y_beta                      # 形状: (batch, seqlen, nheads, headdim)
        ssm_state = state_gamma + state_beta      # 形状: (batch, nheads, headdim, d_state)

        # 5.跳跃连接: y = y + D * x 
        y = y + x * self.D.unsqueeze(-1)          # 形状: (batch, seqlen, nheads, headdim)

        # 6.门控
        y = rearrange(y, "b l h p -> b l (h p)")  # 形状: (batch, seqlen, d_inner)
        y = y * silu(z)                           # 形状: (batch, seqlen, d_inner)

        # 7.输出投影
        y = self.out_proj(y)                      # 形状: (batch, seqlen, d_model)

        # 8.构建推理缓存
        last_Bx = torch.einsum(                   # 形状: (batch, nheads, headdim, d_state)
            "bhn, bhp -> bhpn",
            B[:, -1], x[:, -1],
        )  

    # 获取最后一个时间步的累积角度
    last_cum_angle = cum_angles[:, -1:]           # 形状：(batch, 1, nheads, d_state//2)
    
    # 创建新的推理缓存
    h_new = InferenceCache(
        ssm_state=ssm_state,                      # 形状: (batch, nheads, headdim, d_state)
        prev_Bx=last_Bx,                          # 形状: (batch, nheads, headdim, d_state)
        cum_angle=last_cum_angle.squeeze(1),      # 形状: (batch, nheads, d_state//2)
    )
    
    #返回输出和新推理缓存
    return y, h_new

In [48]:
Mamba3.forward = Mamba3forward

**Mamba-3 的自回归推理**   
每个新 token 的生成依赖于之前所有生成的 token；必须按顺序一个接一个地生成，不能并行；新 token 必须基于最新的上下文状态。每次 `step` 调用的时间复杂度为 `O(1)`

**与 Transformer 的对比**
| 模型 | 推理时间复杂度 | 内存使用 | 长序列表现 |
|------|--------------|---------|-----------|
| Transformer | O(L²) | O(L²) | 随着序列增长速度显著变慢 |
| Mamba-3 | O(1) | O(1) | 速度稳定，不受序列长度影响 |



**step**
1. 升维：(batch, seqlen:1, d_model) -> (batch, d_model) -> (batch, d_in_proj)
2. 分割输入投影
    - z: (batch, d_inner)
    - x: (batch, d_inner)
    - B: (batch, bc_dim)
    - C: (batch, bc_dim)
    - dt: (batch, nheads)
    - lam: (batch, nheads)
    - theta: (batch, d_state//2)
3. 初始化 dt 和 lam ，限定范围
4. BC归一化，稳定训练
5. 数据依赖
6. 准备ssd参数：alpha：状态衰减系数，beta：前一个输入对当前状态影响，gamma：当前输入对当前状态影响
7. 调整输入x的形状适配ssd

In [51]:
def Mamba3step(self, u: Tensor, h: InferenceCache) -> tuple[Tensor, InferenceCache]:
    """
    单 token 推理步骤实现 Mamba-3 递归 (Eq. 9):
        h_t = α_t h_{t-1} + β_t B̄_{t-1} x_{t-1} + γ_t B̄_t x_t
        y_t = C̄_t^T h_t

    参数
        u: (batch, 1, d_model)
        h: 当前推理缓存 (ssm_state, prev_Bx, cum_angle)

    返回 (y, h_new)
        y: (batch, 1, d_model)
        h_new: 更新的推理缓存
        """
    # seqlen = 1
    assert u.shape[1] == 1, "每次推理步骤只能解码一个 token"
    args = self.args

    # 获取 A 的负对角线（同forward）
    A = -torch.exp(self.A_log)                   # (nheads,)

    # 挤压第 1 维：(batch, 1, d_model) -> (batch, d_model)
    # 1.上投影，升维： (batch, d_model) -> (batch, d_in_proj)
    proj = self.in_proj(u.squeeze(1))  
    
    # 2.分割投影结果
    z, x, B, C, dt, lam, theta = torch.split(
        proj,
        [
            args.d_inner,
            args.d_inner,
            self.bc_dim,
            self.bc_dim,
            args.nheads,
            args.nheads,
            args.d_state // 2,
        ],
        dim=-1,)

    # 3.离散化 （同forward）
    dt = F.softplus(dt + self.dt_bias)           # (batch, nheads)
    lam = torch.sigmoid(lam)                     # (batch, nheads)

    # 4. QK-Norm （同forward）
    B = self.B_norm(B)                           # (batch, bc_dim)
    C = self.C_norm(C)                           # (batch, bc_dim)

    # 5.数据依赖的 RoPE  （同forward） 形状: (batch, nheads, d_state//2)
    # 更新累积角度: Θ_{t}[h,j] = Θ_{t-1}[h,j] − Δ_t[h] * θ_t[j]
    raw_angle = (
        dt.unsqueeze(-1)                         # (batch, nheads, 1)
        * theta.unsqueeze(1)                     # (batch, 1, d_state//2)
    )

    # h.cum_angle：从序列开始到当前位置的累积旋转角度
    # raw_angle：当前步骤的旋转角度增量
    # new_cum_angle：表示从序列开始到下一个位置的累积旋转角度
    new_cum_angle = h.cum_angle - raw_angle      # (batch, nheads, d_state//2)

    # 6.梯形系数 （同forward）
    #   α_t = exp(Δ_t * A_t)                     — 衰减
    #   β_t = (1 − λ_t) * Δ_t * exp(Δ_t * A_t)   — 左端点 (前一个输入)
    #   γ_t = λ_t * Δ_t                          — 右端点 (当前输入)
    
    # dA = dt * rearrange(A, "h -> 1, h")
    dA = dt * A                                  # (batch, nheads)
    alpha = torch.exp(dA)  # 衰减
    beta = (1 - lam) * dt * alpha  # 左端点系数
    gamma = lam * dt  # 右端点系数

    # 7.重塑 x: (batch, d_inner) -> (batch, nheads, headdim) 
    x = rearrange(x, "b (h p) -> b h p", p=args.headdim) 

    if args.use_mimo:
        R = args.mimo_rank

        # ── 重塑 B, C: (batch, d_state*R) → (batch, d_state, R) ──
        B = rearrange(B, "b (n r) -> b n r", r=R)    # 形状: (batch, d_state, R)
        C = rearrange(C, "b (n r) -> b n r", r=R)    # 形状: (batch, d_state, R)
        
        # 广播到头部 + 偏置: (batch, 1, d_state, R) + (nheads, d_state, R)
        B = B.unsqueeze(1) + self.B_bias             # 形状: (batch, nheads, d_state, R)
        C = C.unsqueeze(1) + self.C_bias             # 形状: (batch, nheads, d_state, R)

        # ── 对每个秩在 d_state 维度上应用 RoPE ──
        B = rearrange(B, "b h n r -> b h r n")       # 形状: (batch, nheads, R, d_state)
        C = rearrange(C, "b h n r -> b h r n")       # 形状: (batch, nheads, R, d_state)

        # new_cum_angle：(batch, nheads, d_state//2) -> (batch, nheads, 1, d_state//2)
        B = apply_rope(B, new_cum_angle.unsqueeze(2)) 
        C = apply_rope(C, new_cum_angle.unsqueeze(2))
        
        B = rearrange(B, "b h r n -> b h n r")       # 形状: (batch, nheads, d_state, R)
        C = rearrange(C, "b h r n -> b h n r")       # 形状: (batch, nheads, d_state, R)

        # ── 将 x 扩展到秩 R ──
        # (batch, nheads, headdim, 1) * (nheads, headdim, R)
        x_mimo = x.unsqueeze(-1) * self.mimo_x_proj  # 形状: (batch, nheads, headdim, R)

        #======================== 递归的ssd ========================
        # ── MIMO 状态更新: B @ X^T 收缩秩 R ──
        BX = torch.einsum("bhnr, bhpr -> bhpn", B, x_mimo)  # 形状: (batch, nheads, P, N)

        # 更新状态  形状: (batch, nheads, headdim, d_state)  同 SISO
        new_ssm_state = (
            h.ssm_state * rearrange(alpha, "b h -> b h 1 1")
            + h.prev_Bx * rearrange(beta, "b h -> b h 1 1")
            + BX * rearrange(gamma, "b h -> b h 1 1")
        )  

        # ── 输出: H^T @ C → 每个头部 (headdim, R) ── yt = C * ht
        y = torch.einsum("bhpn, bhnr -> bhpr", new_ssm_state, C)   # 形状: (batch, nheads, headdim, R)
        #============================================================
        
        # ── 秩-R 空间中的跳跃连接 ──
        # y: (batch, nheads, headdim) -> (batch, nheads, headdim, 1)
        y = y + (x * rearrange(self.D, "h -> 1 h 1")).unsqueeze(-1)

        # ── 秩-R 空间中的门控 ──
        z_heads = rearrange(z, "b (h p) -> b h p", p=args.headdim)  # 形状: (batch, nheads, headdim)
        z_mimo = z_heads.unsqueeze(-1) * self.mimo_z_proj           # 形状: (batch, nheads, headdim, R)
        y = y * silu(z_mimo)

        # ── 下投影秩 ──
        y = (y * self.mimo_down).sum(dim=-1)  # 形状: (batch, nheads, headdim)
        y = rearrange(y, "b h p -> b (h p)")  # 形状: (batch, d_inner)
        y = self.out_proj(y)                  # 形状: (batch, d_model)

        # 创建新的推理缓存
        h_new = InferenceCache(
            ssm_state=new_ssm_state,          # ht: (batch, nheads, headdim, d_state)
            prev_Bx=BX,                       # Bt * xt: (batch, nheads, headdim, d_state)
            cum_angle=new_cum_angle,          # 旋转角度累积: (batch, nheads, d_state//2)
        )

        # (batch, d_model) -> (batch, seqlen:1, d_model)
        return y.unsqueeze(1), h_new

    else:
        # SISO 路径
        # B、C:           (batch,  d_state) -> (batch, 1, d_state) -> (b, nheads, d_state)      
        # B_bias、C_bias: (nheads, d_state) -> (1, nheads, d_state) -> (b, nheads, d_state)
        B = B.unsqueeze(1) + self.B_bias  
        C = C.unsqueeze(1) + self.C_bias  
        
        B = apply_rope(B, new_cum_angle)       # 形状: (batch, nheads, d_state)
        C = apply_rope(C, new_cum_angle)       # 形状: (batch, nheads, d_state)

        #======================== 递归的ssd ========================
        # 计算 Bx 外积 (batch, nheads, headdim, d_state)
        Bx = torch.einsum("bhn, bhp -> bhpn", B, x) 

        # 更新状态
        # 形状: (batch, nheads, headdim, d_state)
        new_ssm_state = (                                     # ht = 以下三行相加
            h.ssm_state * rearrange(alpha, "b h -> b h 1 1")  # αt * ht-1
            + h.prev_Bx * rearrange(beta, "b h -> b h 1 1")   # βt * (Bt−1 * xt-1) 
            + Bx * rearrange(gamma, "b h -> b h 1 1")         # γt * (Bt * xt)
        )  

        # 计算输出 yt = C * ht  形状: (batch, nheads, headdim)
        y = torch.einsum("bhpn, bhn -> bhp", new_ssm_state, C) 
        #============================================================
        
        # 跳跃连接（残差）  每个头部可能需要不同强度的跳跃连接 D可学习并且初始全1
        # 等价于 y = y + rearrange(self.D, "h -> 1 h 1") * x
        y = y + rearrange(self.D, "h -> h 1") * x   # 形状: (batch, nheads, headdim)

        # 门控
        y = rearrange(y, "b h p -> b (h p)")        # 形状: (batch, d_inner)
        y = y * silu(z)                             # 形状: (batch, d_inner)
        
        # 输出投影
        y = self.out_proj(y)                        # 形状: (batch, d_model)

        # 创建新的推理缓存
        h_new = InferenceCache(
            ssm_state=new_ssm_state,                # ht: (batch, nheads, headdim, d_state)
            prev_Bx=Bx,                             # Bt * xt: (batch, nheads, headdim, d_state)
            cum_angle=new_cum_angle,                # 旋转角度累积: (batch, nheads, d_state//2)
        )

        # (batch, d_model) -> (batch, seqlen:1, d_model)
        return y.unsqueeze(1), h_new

In [52]:
Mamba3.step = Mamba3step

## Mamba3LMHeadModel

Mamba3语言模型头部
- 主干
    - 词嵌入层
    - 模型层 列表
        - 归一化层
        - Mamba3块
        - 归一化层
        - 门控线性单元
    - 最终归一化层
- 语言模型头部

In [53]:
class Mamba3LMHeadModel(nn.Module):
    def __init__(self, args: Mamba3Config, device: Device = None):
        super().__init__()
        self.args = args
        self.device = device

        # 模型主干
        self.backbone = nn.ModuleDict(
            dict(
                # 词嵌入层：将 token ID 转换为向量  embedding形状 (vocab_size, d_model)
                embedding=nn.Embedding(args.vocab_size, args.d_model, device=device),
                # 模型层 列表
                layers=nn.ModuleList(
                    [
                        nn.ModuleDict(
                            dict(
                                # Llama 风格: 预归一化 → mixer → 残差连接, 预归一化 → MLP → 残差连接
                                mixer_norm=RMSNorm(args.d_model, device=device),
                                mixer=Mamba3(args, device=device),
                                
                                mlp_norm=RMSNorm(args.d_model, device=device),
                                mlp=SwiGLU(args.d_model, args.d_mlp_inner, device=device),
                            )
                        )
                        for _ in range(args.n_layer)  # 每层都有着四个模块 像transformer的Encoder
                    ]
                ),
                # 最终归一化层
                norm_f=RMSNorm(args.d_model, device=device),
            )
        )
        
        # 语言模型头部，将隐藏状态映射到词汇表大小，预测每个词的概率
        # lm_head形状 (d_model, vocab_size)
        self.lm_head = nn.Linear(
            args.d_model, args.vocab_size, bias=False, device=device
        )
        
        # 嵌入层 和 语言模型头部 的权重共享 (标准做法, 同transformer)
        # PyTorch 的线性层在计算时会自动处理转置
        self.lm_head.weight = self.backbone.embedding.weight

**forward**  
Arguments  
- input_ids: (batch, seqlen) token IDs
- h: 推理步骤的隐藏状态。如果存在，则采用常数时间 (相对于序列长度) 推理路径；input_ids 应具有形状 (batch, 1).

Return (logits, h)  
- logits: (batch, seqlen, vocab_size)
- h: 处理 input_ids 后的更新推理缓存

In [61]:
def Mamba3LMHeadModelforward(
        self, input_ids: LongTensor, h: list[InferenceCache] | list[None] | None = None
    ) -> tuple[LongTensor, list[InferenceCache]]:
    # -> 指定函数返回 一个元祖，第一个是 LongTensor ，第二个是 InferenceCache 的列表类型
    
    seqlen = input_ids.shape[1]

    # 如果没有提供推理缓存，则为每层创建 None
    if h is None:
        h = [None for _ in range(self.args.n_layer)]

    # 词嵌入: (batch, seqlen) -> (batch, seqlen, d_model)
    x = self.backbone.embedding(input_ids)

    #遍历每个Mamba3块
    for i, layer in enumerate(self.backbone.layers):
        # x -> RMSNorm -> Mamba3 -> y
        y, h[i] = layer.mixer(layer.mixer_norm(x), h[i])

        # 残差连接
        x = y + x  

        # x -> RMSNorm -> SwiGLU MLP -> +x (残差连接) -> x
        x = x + layer.mlp(layer.mlp_norm(x))

    # 最终归一化 x: (batch, seqlen, d_model)
    x = self.backbone.norm_f(x)

    # 将输出映射到词表大小 得到每个可能的下一个词的概率
    # (batch, seqlen, d_model) -> (batch, seqlen, vocab_size)
    logits = self.lm_head(x)

    # 切片到seqlen ：确保只返回每个样本的实际序列长度部分，去除填充部分的输出
    # cast : 将 h 转换为 InferenceCache 列表类型
    return logits[:, :seqlen], cast(list[InferenceCache], h)

In [60]:
Mamba3LMHeadModel.forward = Mamba3LMHeadModelforward

**generate**  model.generate的实现
自回归生成，一次产生一个 token。  

| 参数名 | 类型 | 默认值 | 含义 |
|--------|------|--------|------|
| `input_ids` | `LongTensor` | 必填 | 输入的 token ID 序列，形状为 `(seqlen,)`，作为生成的前缀 |
| `max_new_length` | `int` | 20 | 最大生成 token 数量，控制生成序列的长度 |
| `temperature` | `float` | 1.0 | 温度参数，控制生成的随机性。值越大，生成越随机；值越小，生成越确定 |
| `top_k` | `int` | 50 | 限制只从概率最高的 `k` 个 token 中选择，减少生成的随机性 |
| `top_p` | `float` | 1.0 | 核采样参数，控制累积概率。值小于 1.0 时，只考虑累积概率达到 `top_p` 的 token |
| `eos_token_id` | `int` | 0 | 结束 token 的 ID，当生成到这个 token 时停止生成 |

**`返回值`**
- **类型**：`Iterable[tuple[int, list[InferenceCache]]]`
- **含义**：
  - 可迭代对象，每次产生一个新生成的 token ID
  - 同时返回更新后的推理缓存，用于后续的生成步骤
- **用途**：允许逐 token 生成，便于实时显示生成过程


**`生成过程`**

1. **处理前缀**：将输入序列分割为前缀和最后一个 token
2. **初始化**：通过前向传播处理前缀，获取初始推理缓存
3. **自回归生成**：循环生成新 token，每次：
   - 使用 `step` 方法预测下一个 token
   - 应用温度、top-k 和 top-p 采样
   - 检查是否生成了结束 token
   - 更新推理缓存
4. **停止条件**：达到 `max_new_length` 或生成了 `eos_token_id`


In [69]:
def Mamba3LMHeadModelgenerate(
        self,
        input_ids: LongTensor,
        max_new_length: int = 20,
        temperature: float = 1.0,
        top_k: int = 50,
        top_p: float = 1.0,
        eos_token_id: int = 0,
    ) -> Iterable[tuple[int, list[InferenceCache]]]:

    # 分割前缀和最后一个 token
    # prefix (seqlen-1,)  tokens (1,1)
    prefix, tokens = input_ids[:-1], input_ids[-1:].unsqueeze(0)

    # ── 处理提示 ──
    # 前向（非推理）路径的输入长度必须是 chunk_size 的倍数。
    # 我们通过前向处理大部分内容，通过推理路径逐个处理尾部 token。
    n_chunked = (prefix.shape[0] // self.args.chunk_size) * self.args.chunk_size

    # 对这部分前缀使用前向传播（批量处理），获取初始的推理缓存 h
    # 调用HeadModel的forward -> 调用Mamba3的forward -> h为None -> 继续执行Mamba3的forward
    if n_chunked > 0:
        _, h = self(prefix[:n_chunked].unsqueeze(0), None)
    #如果前缀长度为 0，则初始化空的推理缓存
    else:
        h = [
            InferenceCache.alloc(1, self.args, device=self.device)
            for _ in range(self.args.n_layer)
        ]
        
    # 处理剩余前缀 使用推理模式（step 方法）逐个处理
    # 为什么是step：调用HeadModel的forward -> 调用Mamba3的forward -> h不为None -> step()
    for i in range(n_chunked, prefix.shape[0]):
        _, h = self(prefix[i : i + 1].unsqueeze(0), h)

    # 自回归生成循环
    for _ in range(max_new_length):
        # 预测下一个 token
        # tokens: (1,1)
        # out : (batch:1, seqlen:1, vocab_size) 所有词是下一个词的概率
        with torch.no_grad():
            out, h = self(tokens, h)

        # logits：(batch:1, seqlen:1, vocab_size) -> (vocab_size,)
        logits = out[0, -1]

        # 让概率分布 变平坦(temperature>1) 或者 变尖锐(temperature<1)
        if temperature != 1.0:
            logits = logits / temperature
            
        if top_k > 0:
            # 找出所有小于阈值的logits的索引
            indices_to_remove = logits < torch.topk(logits, k=top_k)[0][-1]
            # 将这些索引对应的logits设置为负无穷大, 对应的概率会趋近于0
            logits[indices_to_remove] = -torch.inf

        if top_p < 1.0:
            # 对logits按降序排序
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            # 计算排序后logits的softmax概率，并计算累积概率
            cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            # 找出累积概率超过top_p的索引
            sorted_indices_to_remove = cum_probs > top_p
            # 确保至少保留一个token
            sorted_indices_to_remove[1:] = sorted_indices_to_remove[:-1].clone()
            sorted_indices_to_remove[0] = False
            # 将排序后的索引映射回原始logits的索引
            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            # 将这些索引对应的logits设置为负无穷大
            logits[indices_to_remove] = -torch.inf

        # 对最后一维计算概率分布  (vocab_size,)
        probs = F.softmax(logits, dim=-1)
        # 从概率分布中 随机采样 一个token (概率高的容易被选中)
        next_token = torch.multinomial(probs, num_samples=1)
        # 如果采样到结束token，则终止生成过程
        if next_token.item() == eos_token_id:
            return
        # 将新生成的token作为下一次的输入    
        tokens = next_token.unsqueeze(0)
        # 生成器函数每次调用时执行到 yield 语句就会暂停，并返回一个值，下次调用时从暂停的地方继续执行，而不是重新开始
        # 生成并返回当前token和更新后的隐藏状态
        yield cast(int, next_token.item()), h

In [70]:
Mamba3LMHeadModel.generate = Mamba3LMHeadModelgenerate

## Mamba3完整模型测试

In [82]:
import gc
import time

### 模型结构测试

In [101]:
def demo_architecture():
    device = get_device()
    print(f"╔══════════════════════════════════════════════════════════════╗")
    print(f"║  Mamba-3 Minimal — Architecture Demo                         ║")
    print(f"║  Device: {str(device):<52}║")
    print(f"╚══════════════════════════════════════════════════════════════╝\n")

    # ── 模型配置 ──
    args = Mamba3Config(
        d_model=128,
        n_layer=4,
        d_state=64,
        headdim=32,
        chunk_size=32,
        vocab_size=256,
    )
    model = Mamba3LMHeadModel(args, device=device)
    n_params = sum(p.numel() for p in model.parameters())

    print("模型配置:")
    print(f"  d_model      = {args.d_model}")
    print(f"  n_layer      = {args.n_layer}")
    print(f"  d_state      = {args.d_state}")
    print(f"  d_inner      = {args.d_inner} (expand={args.expand} × d_model)")
    print(f"  nheads       = {args.nheads}")
    print(f"  headdim      = {args.headdim}")
    print(f"  chunk_size   = {args.chunk_size}")
    print(f"  d_mlp_inner  = {args.d_mlp_inner}")
    print(f"  vocab_size   = {args.vocab_size}")
    print(f"  Parameters   = {n_params:,}\n")

    # ── 初始化参数 ──
    for name, p in model.named_parameters():
        if "A_log" in name:
            # 均匀分布初始化，范围 [-4, -1]
            torch.nn.init.uniform_(p, -4, -1)
        elif "D" in name and p.dim() == 1:
            # D 参数 （一维）初始化为 1
            torch.nn.init.ones_(p)
        elif "dt_bias" in name:
            # 均匀分布初始化，范围 [0.001, 0.1] 确保 dt 在合理范围内，避免步长过大导致状态更新不稳定
            torch.nn.init.uniform_(p, 0.001, 0.1)
        elif p.dim() >= 2:
            # 其他二维及以上参数： 正态分布初始化，标准差 0.02
            torch.nn.init.normal_(p, std=0.02)

    # ── Layer Architecture ──
    print("层结构 (Llama风格, 每层):")
    print("  ┌─ RMSNorm ─→ Mamba3 SSM Block ─→ Residual Add")
    print("  │    (pre-norm)    ↓")
    print("  │    Input Projection → split (z, x, B, C, Δ, λ, θ)")
    print("  │    B, C → QK-Norm → +Bias → RoPE rotation")
    print("  │    Trapezoidal SSD (γ term + β term)")
    print("  │    y + D·x (skip) → y · SiLU(z) (gate) → Out Projection")
    print("  │")
    print("  └─ RMSNorm ─→ SwiGLU MLP ─→ Residual Add")
    print(f"       (pre-norm)   W_gate: {args.d_model}→{args.d_mlp_inner}")
    print()

    return model, args

In [102]:
model, args = demo_architecture()

╔══════════════════════════════════════════════════════════════╗
║  Mamba-3 Minimal — Architecture Demo                         ║
║  Device: cuda                                                ║
╚══════════════════════════════════════════════════════════════╝

模型配置:
  d_model      = 128
  n_layer      = 4
  d_state      = 64
  d_inner      = 256 (expand=2 × d_model)
  nheads       = 8
  headdim      = 32
  chunk_size   = 32
  d_mlp_inner  = 512
  vocab_size   = 256
  Parameters   = 1,308,384

层结构 (Llama风格, 每层):
  ┌─ RMSNorm ─→ Mamba3 SSM Block ─→ Residual Add
  │    (pre-norm)    ↓
  │    Input Projection → split (z, x, B, C, Δ, λ, θ)
  │    B, C → QK-Norm → +Bias → RoPE rotation
  │    Trapezoidal SSD (γ term + β term)
  │    y + D·x (skip) → y · SiLU(z) (gate) → Out Projection
  │
  └─ RMSNorm ─→ SwiGLU MLP ─→ Residual Add
       (pre-norm)   W_gate: 128→512



### 前向传播（训练模式，使用分块SSD）

In [103]:
def demo_forward_pass(model, args):
    device = next(model.parameters()).device
    print("─── 前向传播（训练模式）─────────────────────────────")

    batch_size, seq_len = 2, 64
    input_ids = torch.randint(0, args.vocab_size, (batch_size, seq_len), device=device)
    print(f"  Input 形状:  ({batch_size}, {seq_len})")

    t0 = time.perf_counter()
    with torch.no_grad():
        logits, h = model(input_ids)
    dt = time.perf_counter() - t0

    print(f"  Output 形状: {logits.shape}")
    print(f"  用时: {dt*1000:.1f}ms")

    # Verify gradient flow
    model.train()
    input_ids_grad = torch.randint(0, args.vocab_size, (1, 32), device=device)
    logits_grad, _ = model(input_ids_grad)
    loss = logits_grad.sum()
    loss.backward()
    grad_norms = {
        name: p.grad.norm().item()
        for name, p in model.named_parameters()
        if p.grad is not None
    }
    print(f"  Gradient check: {len(grad_norms)} parameters have gradients ✓")
    model.zero_grad()
    model.eval()
    print()
    # return h

In [104]:
demo_forward_pass(model, args)

─── 前向传播（训练模式）─────────────────────────────
  Input 形状:  (2, 64)
  Output 形状: torch.Size([2, 64, 256])
  用时: 11.2ms
  Gradient check: 58 parameters have gradients ✓



### 单token推理（自回归解码）

In [105]:
def demo_inference_step(model, args):
    device = next(model.parameters()).device
    print("─── 单token推理（解码模式） ──────────────────────────────")

    batch_size = 1
    h = [InferenceCache.alloc(batch_size, args, device=device) for _ in range(args.n_layer)]

    #一次处理 10个 token
    t0 = time.perf_counter()
    for t in range(10):
        token = torch.randint(0, args.vocab_size, (batch_size, 1), device=device)
        with torch.no_grad():
            logits, h = model(token, h)
    dt = time.perf_counter() - t0

    print(f"  10 个推理步骤用时: {dt*1000:.1f}ms ({dt*100:.1f}ms/token)")
    print(f"  SSM state shape: {h[0].ssm_state.shape}")
    print(f"  Prev B⊗x shape: {h[0].prev_Bx.shape}")
    print(f"  Cum angle shape: {h[0].cum_angle.shape}")
    print()

In [106]:
demo_inference_step(model, args)

─── 单token推理（解码模式） ──────────────────────────────
  10 个推理步骤用时: 46.7ms (4.7ms/token)
  SSM state shape: torch.Size([1, 8, 32, 64])
  Prev B⊗x shape: torch.Size([1, 8, 32, 64])
  Cum angle shape: torch.Size([1, 8, 32])



### 前向传播与逐步骤推理的一致性验证

In [109]:
def demo_consistency(model, args):
    device = next(model.parameters()).device
    print("─── 前向传播（分块SSD）与逐步骤推理（step）的一致性验证 ───────────────────────")

    torch.manual_seed(42)
    seq_len = 64
    test_seq = torch.randint(0, args.vocab_size, (1, seq_len), device=device)

    with torch.no_grad():
        # Chunked forward (training path)
        logits_fwd, _ = model(test_seq)

        # Step-by-step (inference path)
        h_step = [InferenceCache.alloc(1, args, device=device) for _ in range(args.n_layer)]
        logits_list = []
        for t in range(seq_len):
            out, h_step = model(test_seq[:, t:t+1], h_step)
            logits_list.append(out)
        logits_seq = torch.cat(logits_list, dim=1)

    max_diff = (logits_fwd - logits_seq).abs().max().item()
    mean_diff = (logits_fwd - logits_seq).abs().mean().item()

    print(f"  Sequence length: {seq_len}")
    print("   (差异越小越好)")
    print(f"  Max absolute difference 最大差异:  {max_diff:.6e}")
    print(f"  Mean absolute difference 平均差异: {mean_diff:.6e}")
    if max_diff < 1e-2:
        print("  ✓ Outputs are consistent!")
    else:
        print("  ⚠ Outputs differ — check implementation.")
    print()

In [110]:
demo_consistency(model, args)

─── 前向传播（分块SSD）与逐步骤推理（step）的一致性验证 ───────────────────────
  Sequence length: 64
   (差异越小越好)
  Max absolute difference 最大差异:  2.592802e-06
  Mean absolute difference 平均差异: 3.768272e-07
  ✓ Outputs are consistent!



### 训练循环演示

In [112]:
def demo_training_loop(model, args, n_steps=50):
    device = next(model.parameters()).device
    print("─── 训练循环演示 ───────────────────────────────────────")

    # 为模型创建一个简单的重复模式以供学习
    torch.manual_seed(123)
    pattern = torch.randint(0, args.vocab_size, (8,), device=device)
    train_data = pattern.repeat(args.chunk_size * 2 // len(pattern) + 1)[:args.chunk_size * 2]
    train_data = train_data.unsqueeze(0)  # (1, seq_len)

    # AdamW优化器
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    model.train()

    print(f"  长度重复模式训练 {len(pattern)}")
    print(f"  序列长度: {train_data.shape[1]}")
    print(f"  训练步数: {n_steps}")
    print()

    losses = []
    t0 = time.perf_counter()
    for step in range(n_steps):
        optimizer.zero_grad()
        logits, _ = model(train_data)

        # Next-token 预测损失： 交叉熵损失是多分类任务的标准损失函数，适合处理自回归序列预测
        loss = F.cross_entropy(
            logits[:, :-1].reshape(-1, args.vocab_size),
            train_data[:, 1:].reshape(-1),
        )
        loss.backward()
        # 梯度裁剪操作，用于防止训练过程中的梯度爆炸问题
        # 计算模型所有参数梯度的 L2 范数（即所有参数梯度的平方和的平方根）
        # 如果梯度范数超过设定的阈值（这里是 1.0），则按比例缩小所有梯度
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        losses.append(loss.item())
        if step % 10 == 0 or step == n_steps - 1:
            print(f"  Step {step:3d} | Loss: {loss.item():.4f}")

    dt = time.perf_counter() - t0
    print(f"\n  训练耗时: {dt:.1f}s ({dt/n_steps*1000:.0f}ms/step)")
    print(f"  初始 loss: {losses[0]:.4f} → 最终 loss: {losses[-1]:.4f}")
    if losses[-1] < losses[0]:
        print("  ✓ Model is learning!")
    else:
        print("  ⚠ Loss did not decrease.")

    model.eval()
    print()

In [113]:
demo_training_loop(model, args)

─── 训练循环演示 ───────────────────────────────────────
  长度重复模式训练 8
  序列长度: 64
  训练步数: 50

  Step   0 | Loss: 5.6382
  Step  10 | Loss: 2.4245
  Step  20 | Loss: 1.0930
  Step  30 | Loss: 0.2914
  Step  40 | Loss: 0.0721
  Step  49 | Loss: 0.0310

  训练耗时: 1.8s (35ms/step)
  初始 loss: 5.6382 → 最终 loss: 0.0310
  ✓ Model is learning!



### 自回归生成

In [114]:
def demo_generation(model, args):
    device = next(model.parameters()).device
    print("─── 自回归生成 ────────────────────────────────")

    prompt = torch.randint(0, args.vocab_size, (5,), device=device)
    print(f"  提示词: {prompt.tolist()}")
    print(f"  Generating 20 tokens...")

    generated = prompt.tolist()
    for token, h in model.generate(prompt, max_new_length=20, temperature=0.8, top_k=10):
        generated.append(token)

    print(f"  生成: {generated}")
    print(f"  总长度: {len(generated)} tokens")
    print()

In [115]:
demo_generation(model, args)

─── 自回归生成 ────────────────────────────────
  提示词: [78, 125, 132, 103, 31]
  Generating 20 tokens...
  生成: [78, 125, 132, 103, 31, 248, 26, 5, 120, 40, 29, 220, 149, 248, 26, 5, 120, 40, 29, 220, 149, 248, 26, 5, 120]
  总长度: 25 tokens



### GPU显存分析

In [126]:
def demo_mps_memory(model, args):
    device = next(model.parameters()).device
    print("─── GPU显存分析 ─────────────────────────────────────")

    # --- Set up backend-specific functions ---
    if device.type == "mps":
        sync = torch.mps.synchronize
        get_alloc = lambda: torch.mps.current_allocated_memory() / (1024 ** 2)
        get_reserved = lambda: torch.mps.driver_allocated_memory() / (1024 ** 2)
        clear = torch.mps.empty_cache
        backend = "MPS (Apple Silicon)"
    elif device.type == "cuda":
        sync = lambda: torch.cuda.synchronize(device)
        get_alloc = lambda: torch.cuda.memory_allocated(device) / (1024 ** 2)
        get_reserved = lambda: torch.cuda.memory_reserved(device) / (1024 ** 2)
        clear = torch.cuda.empty_cache
        backend = "CUDA"
    else:
        print("  已跳过 — 需要 MPS or CUDA for GPU memory 分析.\n")
        return

    print(f"  Backend: {backend}")

    gc.collect()
    clear()
    sync()
    baseline = get_alloc()
    reserved = get_reserved()
    print(f"  模型占用（已分配）:             {baseline:.2f} MB")
    print(f"  Driver reserved 驱动程序保留:   {reserved:.2f} MB")
    print()

    # --- Sweep chunk_size to show memory scaling ---
    seq_len = 1024
    original_cs = args.chunk_size

    print(f"  内存与 chunk_size 关系 (batch=1, seq_len={seq_len}):")
    print()
    print(f"  {'chunk_size':>12} | {'已分配':>8} | {'与基准的差值':>10}")
    print(f"  {'-'*12}-+-{'-'*11}-+-{'-'*12}")

    for cs in [16, 32, 64, 128, 256]:
        if seq_len % cs != 0:
            continue

        args.chunk_size = cs
        gc.collect()
        clear()
        sync()

        input_ids = torch.randint(0, args.vocab_size, (1, seq_len), device=device)
        with torch.no_grad():
            logits, _ = model(input_ids)
        sync()

        peak = get_alloc()
        delta = peak - baseline

        del logits, input_ids
        gc.collect()
        clear()
        sync()

        tag = " ← default" if cs == original_cs else ""
        print(f"  {cs:>12} | {peak:>8.2f} MB | {delta:>9.2f} MB{tag}")

    args.chunk_size = original_cs
    print()
    print("  更大的 chunk_size → 更多内存（O(Q²) 块内注意力）。")
    print("  更小的 chunk_size → 更少内存，更多块间开销。")
    print()

In [127]:
demo_mps_memory(model, args)

─── GPU显存分析 ─────────────────────────────────────
  Backend: CUDA
  模型占用（已分配）:             32.75 MB
  Driver reserved 驱动程序保留:   60.00 MB

  内存与 chunk_size 关系 (batch=1, seq_len=1024):

    chunk_size |      已分配 |     与基准的差值
  -------------+-------------+-------------
            16 |    38.25 MB |      5.51 MB
            32 |    38.25 MB |      5.51 MB ← default
            64 |    38.25 MB |      5.51 MB
           128 |    38.25 MB |      5.51 MB
           256 |    38.25 MB |      5.51 MB

  更大的 chunk_size → 更多内存（O(Q²) 块内注意力）。
  更小的 chunk_size → 更少内存，更多块间开销。



### MIMO（多输入多输出）前向传播

In [132]:
def demo_mimo():
    """Demonstrate the MIMO (Multiple Input Multiple Output) forward pass."""
    device = get_device()
    print("─── MIMO（多输入多输出）前向传播 ───────────────────────")
    
    # ── Model Configuration ──
    # MIMO adds a rank-R dimension to B and C, enabling higher expressivity
    # without increasing the state size (d_state).
    rank = 4
    args = Mamba3Config(
        d_model=128,
        n_layer=1,
        d_state=64,
        headdim=32,
        chunk_size=32,
        vocab_size=256,
        use_mimo=True,
        mimo_rank=rank,
    )
    model = Mamba3LMHeadModel(args, device=device)
    n_params = sum(p.numel() for p in model.parameters())

    print(f"  Configuration: d_model={args.d_model}, d_state={args.d_state}, MIMO Rank={rank}")
    print(f"  Parameters:    {n_params:,} (vs SISO ~86k)")

    # ── Forward Pass ──
    batch_size, seq_len = 2, 64
    input_ids = torch.randint(0, args.vocab_size, (batch_size, seq_len), device=device)
    
    t0 = time.perf_counter()
    with torch.no_grad():
        output, _ = model(input_ids)
    t1 = time.perf_counter()

    print(f"  Input shape:   ({batch_size}, {seq_len})")
    print(f"  Output shape:  {tuple(output.shape)}")
    print(f"  Latency:       {(t1 - t0) * 1000:.2f} ms")
    print("  Status:        MIMO Forward Pass Successful ✓")
    print()

In [133]:
demo_mimo()

─── MIMO（多输入多输出）前向传播 ───────────────────────
  Configuration: d_model=128, d_state=64, MIMO Rank=4
  Parameters:    407,448 (vs SISO ~86k)
  Input shape:   (2, 64)
  Output shape:  (2, 64, 256)
  Latency:       3.16 ms
  Status:        MIMO Forward Pass Successful ✓



## 文本生成测试

### 工具准备

**需要安装tiktoken**  
tiktoken 是 OpenAI 开发的 tokenizer 库 ，用于将文本转换为 token ID 序列，这是大语言模型处理文本的基础步骤  
**主要功能：**
1. 文本编码 ：将文本字符串转换为 token ID 序列
   - 例如： "Hello world" → [15496, 2159]
2. 文本解码 ：将 token ID 序列转换回文本
   - 例如： [15496, 2159] → "Hello world"
3. 支持多种模型 ：
   - GPT-2（50,257 个 token）
   - GPT-3
   - GPT-4
   - Claude 等

In [138]:
# pip install tiktoken

In [139]:
pip show tiktoken

Name: tiktoken
Version: 0.12.0
Summary: tiktoken is a fast BPE tokeniser for use with OpenAI's models
Home-page: https://github.com/openai/tiktoken
Author: Shantanu Jain
Author-email: shantanu@openai.com
License: MIT License

Copyright (c) 2022 OpenAI, Shantanu Jain

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICUL

In [151]:
def create_toy_model(
    d_model: int = 128,
    n_layer: int = 4,
    vocab_size: int = 256,
    device: Device = None,
    use_mimo: bool = False,
    mimo_rank: int = 4,
) -> Mamba3LMHeadModel:
    """Create a small Mamba-3 model for testing and debugging.

    Default configuration: ~3M parameters, suitable for 18GB M3 MacBook.
    Set use_mimo=True to create a MIMO variant with the specified rank.
    """
    if device is None:
        device = get_device()
    args = Mamba3Config(
        d_model=d_model,
        n_layer=n_layer,
        d_state=64,       # Smaller state for toy model
        headdim=32,        # Smaller heads
        chunk_size=32,
        vocab_size=vocab_size,
        use_mimo=use_mimo,
        mimo_rank=mimo_rank,
    )
    model = Mamba3LMHeadModel(args, device=device)
    # 初始化参数
    for name, p in model.named_parameters():
        if "A_log" in name:
            nn.init.uniform_(p, -4, -1)    # A is negative, log(A) ∈ [-4, -1]
        elif "D" in name and p.dim() == 1:
            nn.init.ones_(p)
        elif "dt_bias" in name:
            nn.init.uniform_(p, 0.001, 0.1)
        elif "B_bias" in name or "C_bias" in name:
            pass  # 已全 1初始化
        elif "mimo" in name:
            pass  # 已全 1或 1/R 初始化
        elif p.dim() >= 2:
            nn.init.normal_(p, std=0.02)
    return model

### 未训练的模型

In [152]:
import tiktoken

# 获取设备
device = get_device()
# 分词器
enc = tiktoken.get_encoding("gpt2")  # 50,257 tokens

print("=" * 62)
print("  真实词汇生成测试 (使用GPT-2 Tokenizer)")
print("=" * 62)
print(f"  Device: {device}")
print(f"  Vocab:  {enc.n_vocab:,} tokens (GPT-2)")
print()

# ── Create model sized for real vocabulary ──
print("创建Mamba3模型 (d_model=256, 4 layers)...")
model = create_toy_model(
    d_model=256,
    n_layer=4,
    vocab_size=enc.n_vocab,
    device=device,
)

# 设置为评估模式
model.eval()
# 获取总参数量
n_params = sum(p.numel() for p in model.parameters())
print(f"  总参数量: {n_params:,}")
print()

# 编码 提示词
prompt_text = "The secret to machine learning is"
# generate() expects a 1-D tensor (no batch dimension)
input_ids = torch.tensor(enc.encode(prompt_text), device=device)

print(f"提示词: \"{prompt_text}\"")
print(f"Tokens: {input_ids.tolist()} ({len(input_ids)} tokens)")
print()

# ── Generate tokens using O(1) inference cache ──
print("生成 (未训练的模型 → 胡言乱语)...")
print("─" * 50)
print(prompt_text, end="")

with torch.no_grad():
    for token, _ in model.generate(
        input_ids,
        max_new_length=30,
        temperature=0.8,
        top_k=50,
    ):
        # Defensive decoding: padded vocab may produce out-of-range token IDs
        if token < enc.n_vocab:
            print(enc.decode([token]), end="", flush=True)
        else:
            print(f"[{token}]", end="", flush=True)

print()
print("─" * 50)
print()
print("✓ Inference cache and generation loop work with real vocabularies!")

  真实词汇生成测试 (使用GPT-2 Tokenizer)
  Device: cuda
  Vocab:  50,257 tokens (GPT-2)

创建Mamba3模型 (d_model=256, 4 layers)...
  总参数量: 17,009,600

提示词: "The secret to machine learning is"
Tokens: [464, 3200, 284, 4572, 4673, 318] (6 tokens)

生成 (未训练的模型 → 胡言乱语)...
──────────────────────────────────────────────────
The secret to machine learning isillerne Richardsonaepernick annoying Blair occult tale Wish Carlo Aloneampton schizophrenarms prompting effortsFootIENT pounded escalation Detentionmatchedurbed shellingMatrix documents SoldiersÃÂÃÂÃÂÃÂ employercoins
──────────────────────────────────────────────────

✓ Inference cache and generation loop work with real vocabularies!


### 训练模型 + 生成测试

In [157]:
import tiktoken

# 获取设备
device = get_device()
# 分词器
enc = tiktoken.get_encoding("gpt2")  # 50,257 tokens

print("=" * 62)
print("  真实词汇生成测试 (使用GPT-2 Tokenizer)")
print("=" * 62)
print(f"  Device: {device}")
print(f"  Vocab:  {enc.n_vocab:,} tokens (GPT-2)")
print()

# ── Create model sized for real vocabulary ──
print("创建Mamba3模型 (d_model=256, 4 layers)...")
model = create_toy_model(
    d_model=256,
    n_layer=4,
    vocab_size=enc.n_vocab,
    device=device,
)

# 获取总参数量
n_params = sum(p.numel() for p in model.parameters())
print(f"  总参数量: {n_params:,}")
print()

  真实词汇生成测试 (使用GPT-2 Tokenizer)
  Device: cuda
  Vocab:  50,257 tokens (GPT-2)

创建Mamba3模型 (d_model=256, 4 layers)...
  总参数量: 17,009,600



准备训练数据

In [158]:
#=============================准备训练数据=============================
print("Generating training data...")
# 5 条训练文本
training_texts = [
    "Machine learning is a field of artificial intelligence that uses algorithms to learn from data and make predictions.",
    "Deep learning is a subset of machine learning that uses neural networks with many layers to learn complex patterns.",
    "Natural language processing is a branch of AI that focuses on enabling computers to understand and generate human language.",
    "Computer vision is a field of AI that enables computers to interpret and understand the visual world through images and videos.",
    "Reinforcement learning is a type of machine learning where an agent learns to make decisions by interacting with an environment and receiving rewards or penalties."
]

# 编码文本为 token ID
encoded_texts = [enc.encode(text) for text in training_texts]

# 找到最长的序列长度
max_length = max(len(encoded) for encoded in encoded_texts)
print(f"  最长序列长度: {max_length} tokens")
print()

# 获取模型的 chunk_size
chunk_size = model.args.chunk_size
# 计算需要的序列长度，确保是 chunk_size 的整数倍
required_length = ((max_length + chunk_size - 1) // chunk_size) * chunk_size
print(f"  Required sequence length (multiple of chunk_size {chunk_size}): {required_length} tokens")
print()

# 填充所有序列到相同长度
padded_texts = []
for encoded in encoded_texts:
    # 填充 0（通常是 pad token）到 required_length
    padded = encoded + [0] * (required_length - len(encoded))
    padded_texts.append(padded)

# 转换为张量
training_data = torch.tensor(padded_texts, device=device)  # (5, required_length)
n_samples = training_data.shape[0]
print(f"  Training data shape: {training_data.shape}")
print()

Generating training data...
  最长序列长度: 28 tokens

  Required sequence length (multiple of chunk_size 32): 32 tokens

  Training data shape: torch.Size([5, 32])



开始训练

In [159]:
#==============================开始训练==============================
print("Starting training...")
print("─" * 50)

# AdamW优化器
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
model.train()

batch_size = 2
n_steps = 50
losses = []

for step in range(n_steps):
    # 随机选择 batch size 个样本
    indices = torch.randint(0, n_samples, (batch_size,))
    batch_input_ids = training_data[indices]  # (batch_size, max_length)
    
    optimizer.zero_grad()
    logits, _ = model(batch_input_ids)
    
    # Next-token prediction loss
    loss = F.cross_entropy(
        logits[:, :-1].reshape(-1, model.args.vocab_size),
        batch_input_ids[:, 1:].reshape(-1),
    )
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    losses.append(loss.item())
    if step % 10 == 0 or step == n_steps - 1:
        print(f"  Step {step:3d} | Loss: {loss.item():.4f}")

print("─" * 50)
print(f"  Initial loss: {losses[0]:.4f} → Final loss: {losses[-1]:.4f}")
if losses[-1] < losses[0]:
    print("  ✓ Model is learning!")
else:
    print("  ⚠ Loss did not decrease.")
print()

Starting training...
──────────────────────────────────────────────────
  Step   0 | Loss: 10.8352
  Step  10 | Loss: 5.7990
  Step  20 | Loss: 2.0703
  Step  30 | Loss: 0.9791
  Step  40 | Loss: 0.1105
  Step  49 | Loss: 0.0917
──────────────────────────────────────────────────
  Initial loss: 10.8352 → Final loss: 0.0917
  ✓ Model is learning!



生成测试

In [160]:
#==============================生成测试===========================
print("Testing generation...")
print("─" * 50)
model.eval()

# 测试提示词
test_prompt = "Artificial intelligence is"
input_ids = torch.tensor(enc.encode(test_prompt), device=device)

print(f"提示词: \"{test_prompt}\"")
print(f"Generating 20 tokens...")
print(test_prompt, end="")

with torch.no_grad():
    for token, _ in model.generate(
        input_ids,
        max_new_length=20,
        temperature=0.8,
        top_k=50,
    ):
        if token < enc.n_vocab:
            print(enc.decode([token]), end="", flush=True)
        else:
            print(f"[{token}]", end="", flush=True)

print()
print("─" * 50)
print()
print("训练测试完成!")

Testing generation...
──────────────────────────────────────────────────
提示词: "Artificial intelligence is"
Generating 20 tokens...
Artificial intelligence is a branch of AI that focuses on enabling computers to understand and generate human language.
──────────────────────────────────────────────────

训练测试完成!
